In [1]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

In [2]:
pred_data = pd.read_csv('pred_data\CIS_662 _INITIAL_Predictions.csv')

In [3]:
pred_data.head()

,DATE,DAY,FLIGHT NUMBER,ORIGIN,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late
0,4/10/24,WEDNESDAY,UA 1400,ORD,6:52 PM,9:47 PM,NaN,NaN,NaN,NaN
1,4/10/24,WEDNESDAY,AA 3402,ORD,7:59 PM,10:52 PM,NaN,NaN,NaN,NaN
2,4/10/24,WEDNESDAY,B6 116,JFK,1:33 PM,2:50 PM,NaN,NaN,NaN,NaN
3,4/10/24,WEDNESDAY,DL 5182,JFK,2:55 PM,4:21 PM,NaN,NaN,NaN,NaN
4,4/10/24,WEDNESDAY,WN 5285,MCO,11:05 AM,1:45 PM,NaN,NaN,NaN,NaN


In [4]:
pred_data.dtypes

DATE                                  object
DAY                                   object
FLIGHT NUMBER                         object
ORIGIN                                object
DEPARTURE TIME                        object
ARRIVAL TIME                          object
ARRIVAL STATUS                       float64
ARRIVAL STATUS_Prev_flight_early     float64
ARRIVAL STATUS_Prev_flight_ontime    float64
ARRIVAL STATUS_Prev_flight_late      float64
dtype: object

In [5]:
pred_data['DATE'] = pd.to_datetime( pred_data['DATE'],format ="%m/%d/%y")
pred_data['DAY'] = pred_data['DATE'].dt.day_of_week.astype('object')
pred_data['Carrier_Code'] = pred_data['FLIGHT NUMBER'].str[0:2].astype('object')
pred_data['DEPARTURE TIME'] = pd.to_datetime(pred_data['DEPARTURE TIME'], format = '%I:%M %p')
pred_data['dep_hour']= pred_data['DEPARTURE TIME'].dt.hour.astype('object')
pred_data['dep_min'] = pred_data['DEPARTURE TIME'].dt.minute.astype('object')
pred_data['ARRIVAL TIME'] = pd.to_datetime(pred_data['ARRIVAL TIME'], format = '%I:%M %p')
pred_data['arr_hour']= pred_data['ARRIVAL TIME'].dt.hour.astype('object')
pred_data['arr_min'] = pred_data['ARRIVAL TIME'].dt.minute.astype('object')
pred_data.rename(columns = {'ORIGIN' : 'Origin_Airport'},inplace =True)
pred_data.head()

,DATE,DAY,FLIGHT NUMBER,Origin_Airport,DEPARTURE TIME,ARRIVAL TIME,ARRIVAL STATUS,ARRIVAL STATUS_Prev_flight_early,ARRIVAL STATUS_Prev_flight_ontime,ARRIVAL STATUS_Prev_flight_late,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min
0,2024-04-10,2,UA 1400,ORD,1900-01-01 18:52:00,1900-01-01 21:47:00,NaN,NaN,NaN,NaN,UA,18,52,21,47
1,2024-04-10,2,AA 3402,ORD,1900-01-01 19:59:00,1900-01-01 22:52:00,NaN,NaN,NaN,NaN,AA,19,59,22,52
2,2024-04-10,2,B6 116,JFK,1900-01-01 13:33:00,1900-01-01 14:50:00,NaN,NaN,NaN,NaN,B6,13,33,14,50
3,2024-04-10,2,DL 5182,JFK,1900-01-01 14:55:00,1900-01-01 16:21:00,NaN,NaN,NaN,NaN,DL,14,55,16,21
4,2024-04-10,2,WN 5285,MCO,1900-01-01 11:05:00,1900-01-01 13:45:00,NaN,NaN,NaN,NaN,WN,11,5,13,45


In [6]:
pred_data.dtypes

DATE                                 datetime64[ns]
DAY                                          object
FLIGHT NUMBER                                object
Origin_Airport                               object
DEPARTURE TIME                       datetime64[ns]
ARRIVAL TIME                         datetime64[ns]
ARRIVAL STATUS                              float64
ARRIVAL STATUS_Prev_flight_early            float64
ARRIVAL STATUS_Prev_flight_ontime           float64
ARRIVAL STATUS_Prev_flight_late             float64
Carrier_Code                                 object
dep_hour                                     object
dep_min                                      object
arr_hour                                     object
arr_min                                      object
dtype: object

In [7]:
pred_data['arr_day'] = pred_data['DAY']
pred_data['dep_day'] = pred_data['DAY']
pred_data.drop(columns = ['DAY','DEPARTURE TIME','ARRIVAL TIME','ARRIVAL STATUS_Prev_flight_early',
                         'ARRIVAL STATUS_Prev_flight_ontime','ARRIVAL STATUS_Prev_flight_late',
                         'FLIGHT NUMBER','DATE','ARRIVAL STATUS'],inplace = True)



In [8]:
pred_data.head(50)

,Origin_Airport,Carrier_Code,dep_hour,dep_min,arr_hour,arr_min,arr_day,dep_day
0,ORD,UA,18,52,21,47,2,2
1,ORD,AA,19,59,22,52,2,2
2,JFK,B6,13,33,14,50,2,2
3,JFK,DL,14,55,16,21,2,2
4,MCO,WN,11,5,13,45,2,2
5,MCO,B6,13,35,16,25,2,2
6,ORD,UA,18,52,21,47,3,3
7,ORD,AA,19,59,22,52,3,3
8,JFK,B6,13,33,14,50,3,3
9,JFK,DL,14,55,16,21,3,3


In [9]:
pred_data['dep_hour'] = pd.Categorical(pred_data['dep_hour'], categories=[i for i in range(24)])
pred_data['dep_day'] = pd.Categorical(pred_data['dep_day'], categories=[i for i in range(7)])
pred_data['dep_min'] = pd.Categorical(pred_data['dep_min'], categories=[i for i in range(60)])
pred_data['arr_hour'] = pd.Categorical(pred_data['arr_hour'], categories=[i for i in range(24)])
pred_data['arr_day'] = pd.Categorical(pred_data['arr_day'], categories=[i for i in range(7)])
pred_data['arr_min'] = pd.Categorical(pred_data['arr_min'], categories=[i for i in range(60)])
pred_data['Carrier_Code'] = pd.Categorical(pred_data['Carrier_Code'], categories=['AA', 'UA', 'DL', 'B6', 'WN'])
pred_data['Origin_Airport'] = pd.Categorical(pred_data['Origin_Airport'], categories=['ORD', 'JFK', 'MCO'])
pred_data=pred_data[['Carrier_Code', 'dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour',
       'arr_day', 'dep_min', 'arr_min']]
pred_data.head()
pred_data.columns

,Carrier_Code,dep_hour,dep_day,Origin_Airport,arr_hour,arr_day,dep_min,arr_min
0,UA,18,2,ORD,21,2,52,47
1,AA,19,2,ORD,22,2,59,52
2,B6,13,2,JFK,14,2,33,50
3,DL,14,2,JFK,16,2,55,21
4,WN,11,2,MCO,13,2,5,45


Index(['Carrier_Code', 'dep_hour', 'dep_day', 'Origin_Airport', 'arr_hour',
       'arr_day', 'dep_min', 'arr_min'],
      dtype='object')